In [ ]:
import numpy as np
from tensorflow import keras
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.io import savemat
from scipy.io import loadmat

In [ ]:
project_path = '../surrogate-training/'
import sys
sys.path.append(project_path)
from Unet import create_vae
# Build the model
vae_model = create_vae()
vae_model.summary()

In [ ]:
# load training data
perm_CSS = loadmat(project_path + 'training_data.mat')['perm_CSS']
perm_CSC = loadmat(project_path +'training_data.mat')['perm_CSC']
perm_SCC = loadmat(project_path +'training_data.mat')['perm_SCC']
perm_SCS = loadmat(project_path +'training_data.mat')['perm_SCS']
perm_train = np.concatenate((perm_CSS,perm_CSC,perm_SCC,perm_SCS),axis = -1)
perm_train = np.reshape(np.log10(perm_train), (100, 100, 2, -1))
perm_train = np.flip(perm_train, axis=0)
perm_train = np.transpose(perm_train, [3, 0, 1, 2])
print('train input shape:',perm_train.shape)

Perm_CSS = loadmat(project_path +'training_data.mat')['Perm_CSS']
Perm_CSC = loadmat(project_path +'training_data.mat')['Perm_CSC']
Perm_SCC = loadmat(project_path +'training_data.mat')['Perm_SCC']
Perm_SCS = loadmat(project_path +'training_data.mat')['Perm_SCS']
Perm_train = np.concatenate((Perm_CSS,Perm_CSC,Perm_SCC,Perm_SCS),axis = -1)
Perm_train = np.log10(Perm_train[[0,2],:].T)
print('train output shape:',Perm_train.shape)

# load testing data
perm_CSS = loadmat(project_path +'validation_data.mat')['perm_CSS']
perm_CSC = loadmat(project_path +'validation_data.mat')['perm_CSC']
perm_SCC = loadmat(project_path +'validation_data.mat')['perm_SCC']
perm_SCS = loadmat(project_path +'validation_data.mat')['perm_SCS']
perm_test = np.concatenate((perm_CSS,perm_CSC,perm_SCC,perm_SCS),axis = -1)
perm_test = np.reshape(np.log10(perm_test), (100, 100, 2, -1))
perm_test = np.flip(perm_test, axis=0)
perm_test = np.transpose(perm_test, [3, 0, 1, 2])
print('test input shape:',perm_test.shape)

Perm_CSS = loadmat(project_path +'validation_data.mat')['Perm_CSS']
Perm_CSC = loadmat(project_path +'validation_data.mat')['Perm_CSC']
Perm_SCC = loadmat(project_path +'validation_data.mat')['Perm_SCC']
Perm_SCS = loadmat(project_path +'validation_data.mat')['Perm_SCS']
Perm_test = np.concatenate((Perm_CSS,Perm_CSC,Perm_SCC,Perm_SCS),axis = -1)
Perm_test = np.log10(Perm_test[[0,2],:].T)
print('test output shape:',Perm_test.shape)

In [ ]:
def min_max_scale_X(X):
    X_min = np.min(X,axis = (0,1,2))
    X_max = np.max(X,axis = (0,1,2))
    X_scaled = (X-X_min)/(X_max-X_min)-0.5
    return X_scaled, X_min,X_max

def min_max_scale_back_X(X_scaled,X_min,X_max):
    X = (X_scaled+0.5)*(X_max-X_min)+X_min
    return X

def min_max_scale_Y(Y):
    Y_min = np.min(Y,axis = 0)
    Y_max = np.max(Y,axis = 0)
    Y_scaled = (Y-Y_min)/(Y_max-Y_min)-0.5
    return Y_scaled, Y_min,Y_max

def min_max_scale_back_Y(Y_scaled,Y_min,Y_max):
    Y = (Y_scaled+0.5)*(Y_max-Y_min)+Y_min
    return Y

def padding_128(X):
  X_pad = np.zeros((X.shape[0],128,128,X.shape[3]))
  for i in range(X.shape[0]):
    X_pad[i,13:13+X.shape[1],13:13+X.shape[2],:] = X[i,:,:,:]
  return X_pad

In [ ]:
Y_train,Y_min,Y_max = min_max_scale_Y(Perm_train)
X_train,X_min,X_max = min_max_scale_X(perm_train)
Y_test = (Perm_test-Y_min)/(Y_max-Y_min)-0.5
X_test = (perm_test-X_min)/(X_max-X_min)-0.5

X_train = padding_128(X_train)
X_test = padding_128(X_test)
print("X train size", X_train.shape)
print("X test size", X_test.shape)
print("Y train size", Y_train.shape)
print("Y test size", Y_test.shape)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(X_train[0,:,:,0],cmap = 'copper', aspect='equal')
plt.title('kxx')
plt.colorbar(fraction=0.08)

plt.subplot(1,2,2)
plt.imshow(X_train[0,:,:,1],cmap = 'copper', aspect='equal')
plt.title('kzz')
plt.colorbar(fraction=0.08)

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
vae_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),loss='mean_absolute_error')
checkpoint_cb = ModelCheckpoint(project_path +'best_model_mae.weights.h5', monitor='loss', save_best_only=True, save_weights_only=True)
reduce_lr_cb = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=50, min_lr=1e-6, verbose=1)
# Train
history = vae_model.fit(X_train, Y_train, epochs=1000, batch_size=32,validation_data=(X_test, Y_test), callbacks=[checkpoint_cb, reduce_lr_cb])

In [ ]:
# save loss
train_loss = np.array(history.history['loss'])
val_loss = np.array(history.history['val_loss'])
savemat(project_path +'history_mae.mat', {'train_loss': train_loss, 'val_loss': val_loss})

In [ ]:
plt.semilogy(train_loss)
plt.semilogy(val_loss)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train','val'], loc='upper left')
plt.show()